# SRGAN Full Training — Kaggle / Colab

**Project:** CMS Jet Super-Resolution  
**Purpose:** Full training on all available data (no `--max-train-batches` cap).  
Works on Kaggle (GPU T4/P100) and Google Colab (T4).  

## Before running
1. On **Kaggle**: upload the `SRGAN/` folder as a dataset, add the jet parquet dataset and/or CaloChallenge HDF5 dataset as input datasets.
2. On **Colab**: mount Google Drive and set `REPO_ROOT` below, or clone your repo.
3. Select `GPU` as the accelerator (Runtime → Change runtime type).

---

## Cell 1 — Detect platform and set paths

In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── Platform detection ────────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB  = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
print(f'Kaggle: {IS_KAGGLE} | Colab: {IS_COLAB}')

# ── Paths — edit these if your layout differs ─────────────────────────────────
if IS_KAGGLE:
    # Upload SRGAN/ as a Kaggle dataset named 'srgan-codebase'
    REPO_ROOT    = Path('/kaggle/input/srgan-codebase/SRGAN')
    PARQUET_DATA = Path('/kaggle/input/cms-jet-images')          # your parquet dataset
    HDF5_DATA    = Path('/kaggle/input/calochallenge-dataset2')  # CaloChallenge HDF5
    OUT_ROOT     = Path('/kaggle/working/experiments')
elif IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT    = Path('/content/drive/MyDrive/gsoc/task/SRGAN')
    PARQUET_DATA = REPO_ROOT.parent / 'datasets'
    HDF5_DATA    = REPO_ROOT.parent / 'datasets' / 'calochallenge_dataset2'
    OUT_ROOT     = REPO_ROOT / 'experiments' / 'full'
else:
    # Local fallback
    REPO_ROOT    = Path('..').resolve()   # run from notebooks/
    PARQUET_DATA = REPO_ROOT.parent / 'datasets'
    HDF5_DATA    = REPO_ROOT.parent / 'datasets' / 'calochallenge_dataset2'
    OUT_ROOT     = REPO_ROOT / 'experiments' / 'full'

os.chdir(REPO_ROOT)   # all scripts use relative paths from SRGAN/
sys.path.insert(0, str(REPO_ROOT))
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'REPO_ROOT    : {REPO_ROOT}')
print(f'PARQUET_DATA : {PARQUET_DATA}')
print(f'HDF5_DATA    : {HDF5_DATA}')
print(f'OUT_ROOT     : {OUT_ROOT}')

## Cell 2 — Install dependencies

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyarrow', 'h5py', 'matplotlib', 'numpy'], check=True)
print('dependencies OK')

## Cell 3 — Verify GPU and package import

In [ ]:
import torch
from srgan.utils.env import resolve_env

env = resolve_env()
print(f'[env] {env}')

assert env.device.type in ('cuda', 'mps', 'cpu'), 'No accelerator found'
if env.device.type == 'cpu':
    print('WARNING: running on CPU — training will be slow. Enable GPU in runtime settings.')

## Cell 4 — Helper to run a training command with live output

In [ ]:
import subprocess, sys

def run_live(cmd: list[str]) -> int:
    """Run a command and print output line-by-line so progress is visible."""
    print('Running:', ' '.join(str(c) for c in cmd))
    print('-' * 60)
    proc = subprocess.Popen(
        [str(c) for c in cmd],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    print('-' * 60)
    print(f'Exit code: {proc.returncode}')
    return proc.returncode

## Cell 5 — Full training: QCDToGGQQ Parquet dataset

Trains on all ~83K events per epoch for 20 epochs.  
Expected time: **~60–90 min on T4/P100**.  
Skip this cell if you only want to train on CaloChallenge.

In [ ]:
PARQUET_OUT = OUT_ROOT / 'parquet_run_01'

rc = run_live([
    sys.executable, 'train_srgan.py',
    '--dataset-type',    'parquet',
    '--dataset-path',    str(PARQUET_DATA),
    '--epochs',          '20',
    '--batch-size',      '64',
    '--lr',              '2e-4',
    '--d-lr-ratio',      '0.5',
    '--n-critic',        '1',
    '--lambda-l1',       '50',
    '--lambda-physics',  '15',       # increased from 10 — fixes 7.4% under-response
    '--gen-channels',    '64',
    '--gen-blocks',      '8',
    '--hr-size',         '125', '125',
    '--scale-factor',    '2.0',
    '--val-ratio',       '0.33',
    '--seed',            '42',
    '--output-dir',      str(PARQUET_OUT),
])

assert rc == 0, f'Training failed with exit code {rc}'

## Cell 6 — Full training: CaloChallenge HDF5 dataset

Trains on all ~134K showers per epoch for 20 epochs.  
Expected time: **~60–90 min on T4/P100**.  
Skip this cell if you only want to train on parquet.

In [ ]:
HDF5_OUT = OUT_ROOT / 'hdf5_run_01'

rc = run_live([
    sys.executable, 'train_srgan.py',
    '--dataset-type',    'hdf5',
    '--dataset-path',    str(HDF5_DATA),
    '--epochs',          '20',
    '--batch-size',      '64',
    '--lr',              '2e-4',
    '--d-lr-ratio',      '0.5',
    '--n-critic',        '2',        # slow D collapse: update D twice per G step
    '--lambda-l1',       '50',
    '--lambda-physics',  '15',
    '--gen-channels',    '64',
    '--gen-blocks',      '8',
    '--hr-size',         '45', '144',
    '--scale-factor',    '2.0',
    '--val-ratio',       '0.33',
    '--seed',            '42',
    '--output-dir',      str(HDF5_OUT),
])

assert rc == 0, f'Training failed with exit code {rc}'

## Cell 7 — Generate all experiment artifacts (parquet)

In [ ]:
rc = run_live([
    sys.executable, 'save_experiment.py',
    '--run-dir',      str(PARQUET_OUT),
    '--dataset-path', str(PARQUET_DATA),
    '--dataset-type', 'parquet',
    '--hr-size',      '125', '125',
    # no --max-val-samples: use full val split for publication-quality stats
])
assert rc == 0, f'save_experiment failed with exit code {rc}'

## Cell 8 — Generate all experiment artifacts (HDF5)

In [ ]:
rc = run_live([
    sys.executable, 'save_experiment.py',
    '--run-dir',      str(HDF5_OUT),
    '--dataset-path', str(HDF5_DATA),
    '--dataset-type', 'hdf5',
    '--hr-size',      '45', '144',
    '--scale-factor', '2.0',
])
assert rc == 0, f'save_experiment failed with exit code {rc}'

## Cell 9 — Print final summary

In [ ]:
import json

def print_summary(run_dir: Path, label: str) -> None:
    metrics_path = run_dir / 'metrics.jsonl'
    physics_path = run_dir / 'physics_metrics.json'
    corr_path    = run_dir / 'correlation_metrics.json'
    if not metrics_path.exists():
        print(f'{label}: no metrics found at {run_dir}')
        return
    rows   = [json.loads(l) for l in metrics_path.read_text().splitlines() if l.strip()]
    best   = min(rows, key=lambda r: r.get('val_l1', float('inf')))
    phys   = json.loads(physics_path.read_text()) if physics_path.exists() else {}
    corr   = json.loads(corr_path.read_text())    if corr_path.exists()    else {}

    print(f'\n{'='*55}')
    print(f'  {label}')
    print(f'{'='*55}')
    print(f'  Best epoch      : {best["epoch"]} / {len(rows)}')
    print(f'  val_l1          : {best.get("val_l1", "N/A"):.5f}')
    print(f'  val_psnr_norm   : {best.get("val_psnr_norm", "N/A"):.2f} dB')
    print(f'  val_response    : {best.get("val_response", "N/A"):.4f}')
    if phys:
        r = phys['response_gan']
        e = phys['relative_error_gan']
        print(f'  response_gan    : {r["mean"]:.4f} ± {r["std"]:.4f}  (median {r["median"]:.4f})')
        print(f'  |rel error| mean: {e["abs_mean"]:.4f}')
        if phys.get('by_class'):
            for cls, v in phys['by_class'].items():
                print(f'  {cls} response   : {v["response_gan"]["mean"]:.4f} ± {v["response_gan"]["std"]:.4f}')
    if corr:
        print(f'  pearson_r_pixel : {corr.get("pearson_r_pixel", "N/A"):.4f}')
        print(f'  ssim_mean       : {corr.get("ssim_mean", "N/A"):.4f}')

print_summary(PARQUET_OUT, 'QCDToGGQQ Parquet')
print_summary(HDF5_OUT,    'CaloChallenge HDF5')

## Cell 10 — Download artifacts (Kaggle only)

On Kaggle, output files are automatically available via the Output tab.  
On Colab, run this cell to zip and download.

In [ ]:
if IS_COLAB:
    import shutil
    from google.colab import files
    zip_path = '/content/experiments_full.zip'
    shutil.make_archive('/content/experiments_full', 'zip', str(OUT_ROOT.parent), str(OUT_ROOT.name))
    files.download(zip_path)
    print(f'Downloaded: {zip_path}')
elif IS_KAGGLE:
    print('Kaggle: artifacts are in /kaggle/working/experiments')
    print('Access them via the Output tab after the session completes.')
else:
    print(f'Local: artifacts at {OUT_ROOT}')

---

## Quick-start checklist

### Kaggle
1. Create a new notebook → enable GPU
2. Add datasets:
   - Upload `task/SRGAN/` as dataset `srgan-codebase`
   - Add your parquet files dataset
   - Add CaloChallenge Dataset 2 HDF5 dataset
3. Update `PARQUET_DATA` and `HDF5_DATA` paths in Cell 1
4. Run all cells in order
5. Download results from Output tab

### Colab
1. Upload this notebook to Colab
2. Runtime → Change runtime type → T4 GPU
3. Mount Drive (Cell 1 will prompt you)
4. Update `REPO_ROOT` in Cell 1 to point to your SRGAN directory on Drive
5. Run all cells — artifacts download automatically in Cell 10

### Expected outputs after full run
```
experiments/full/
  parquet_run_01/
    checkpoints/best.pt
    metrics.jsonl           (20 epochs)
    physics_metrics.json    (full val split ~46K events)
    correlation_metrics.json
    EXPERIMENT_REPORT.md
    figures/                (22 figures)
  hdf5_run_01/
    checkpoints/best.pt
    metrics.jsonl           (20 epochs)
    physics_metrics.json    (full val split ~66K showers)
    correlation_metrics.json
    EXPERIMENT_REPORT.md
    figures/                (18 figures)
```

### Target metrics for paper-quality results
| Metric | Parquet target | HDF5 target |
|---|---|---|
| val_l1 | < 0.090 | < 0.200 |
| val_response | 0.98 – 1.02 | 0.98 – 1.02 |
| Pearson r | > 0.80 | > 0.93 |
| SSIM | > 0.98 | > 0.94 |